# DX 704 Week 8 Project
This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 2


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return 0

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [4]:
import csv

with open("log-random.tsv", "w", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow([
        "curr_state", "curr_action", "next_reward", "next_state"
    ])

    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
        random_policy, max_steps=32
    ):
        writer.writerow([
            curr_state, curr_action, next_reward, next_state
        ])

        print(
            "CURR STATE", curr_state,
            "ACTION", curr_action,
            "NEXT REWARD", next_reward,
            "NEXT STATE", next_state
        )

CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STATE 123
CURR STATE 123 ACTION 0 NEXT REWARD 0 NEXT STA

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [5]:
# YOUR CHANGES HERE
# Start with all Q-values equal to zero
q_values = {}

alpha = 1.0
gamma = 0.9

q_random_log = []

for episode in range(32):
    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
        random_policy, max_steps=1024
    ):
        old_value = q_values.get((curr_state, curr_action), 0.0)

        best_next_value = max(
            q_values.get((next_state, action), 0.0)
            for action in simulator.get_actions(next_state)
        )

        new_value = old_value + alpha * (
            next_reward + gamma * best_next_value - old_value
        )

        q_values[(curr_state, curr_action)] = new_value

        q_random_log.append([
            curr_state,
            curr_action,
            next_reward,
            next_state,
            old_value,
            new_value
        ])

print("Total updates:", len(q_random_log))
print("Positive Q-values:", sum(value > 0 for value in q_values.values()))

Total updates: 32768
Positive Q-values: 0


In [6]:
with open("q-random.tsv", "w", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow([
        "curr_state",
        "curr_action",
        "next_reward",
        "next_state",
        "old_value",
        "new_value"
    ])
    writer.writerows(q_random_log)

print("Saved q-random.tsv")

Saved q-random.tsv


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [ ]:
# YOUR CHANGES HERE

...

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [7]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
def epsilon_greedy_policy(state, actions):
    if random.random() < 0.25:
        return random.choice(actions)

    best_value = max(
        q_values.get((state, action), 0.0)
        for action in actions
    )

    best_actions = [
        action for action in actions
        if q_values.get((state, action), 0.0) == best_value
    ]

    return random.choice(best_actions)

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [8]:
# YOUR CHANGES HERE

# Reset learning so the log records training from the beginning
q_values = {}

alpha = 1.0
gamma = 0.9
q_greedy_log = []

for episode in range(32):
    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
        epsilon_greedy_policy, max_steps=1024
    ):
        old_value = q_values.get((curr_state, curr_action), 0.0)

        best_next_value = max(
            q_values.get((next_state, action), 0.0)
            for action in simulator.get_actions(next_state)
        )

        new_value = old_value + alpha * (
            next_reward + gamma * best_next_value - old_value
        )

        q_values[(curr_state, curr_action)] = new_value

        q_greedy_log.append([
            curr_state,
            curr_action,
            next_reward,
            next_state,
            old_value,
            new_value
        ])

        # Update and record the successful transition before stopping
        if next_reward > 0:
            break

with open("q-greedy.tsv", "w", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow([
        "curr_state",
        "curr_action",
        "next_reward",
        "next_state",
        "old_value",
        "new_value"
    ])
    writer.writerows(q_greedy_log)

successful_episodes = sum(row[2] > 0 for row in q_greedy_log)

print("Total updates:", len(q_greedy_log))
print("Successful episodes:", successful_episodes, "out of 32")
print("Saved q-greedy.tsv")

Total updates: 18736
Successful episodes: 22 out of 32
Saved q-greedy.tsv


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [9]:
# YOUR CHANGES HERE

policy_greedy = {}

# 16 × 16 positions, each with 8 possible directions
num_states = simulator.resolution ** 2 * 8

for state in range(num_states):
    actions = simulator.get_actions(state)

    best_action = max(
        actions,
        key=lambda action: q_values.get((state, action), 0.0)
    )

    policy_greedy[state] = best_action

with open("policy-greedy.tsv", "w", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow(["state", "action"])

    for state, action in policy_greedy.items():
        writer.writerow([state, action])

print("States in policy:", len(policy_greedy))
print("Saved policy-greedy.tsv")

States in policy: 2048
Saved policy-greedy.tsv


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [10]:
# YOUR CHANGES HERE

# Train a fresh Q-table
q_optimal = {}

gamma = 0.9
num_states = simulator.resolution ** 2 * 8

for training_round in range(200):
    largest_change = 0.0

    for state in range(num_states):
        if state == simulator.terminal_state:
            continue

        for action in simulator.get_actions(state):
            reward, next_state = simulator.get_next_reward_state(
                state, action
            )

            old_value = q_optimal.get((state, action), 0.0)

            if next_state == simulator.terminal_state:
                best_next_value = 0.0
            else:
                best_next_value = max(
                    q_optimal.get((next_state, next_action), 0.0)
                    for next_action in simulator.get_actions(next_state)
                )

            # Q-learning update with alpha = 1
            new_value = reward + gamma * best_next_value

            q_optimal[(state, action)] = new_value

            largest_change = max(
                largest_change, abs(new_value - old_value)
            )

    if largest_change < 1e-12:
        break

# Choose the best learned action for every state
policy_optimal = {}

for state in range(num_states):
    policy_optimal[state] = max(
        simulator.get_actions(state),
        key=lambda action: q_optimal.get((state, action), 0.0)
    )

# Save the required submission file
with open("policy-optimal.tsv", "w", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow(["state", "action"])
    writer.writerows(policy_optimal.items())

print("Training rounds:", training_round + 1)
print("Largest final Q-value change:", largest_change)
print("States in policy:", len(policy_optimal))
print("Saved policy-optimal.tsv")

Training rounds: 13
Largest final Q-value change: 0.0
States in policy: 2048
Saved policy-optimal.tsv


In [11]:
steps_to_goal = []
failed_starts = 0

for initial_state in simulator.initial_states:
    state = initial_state
    visited = set()

    for step in range(1, num_states + 1):
        if state in visited:
            failed_starts += 1
            break

        visited.add(state)

        action = policy_optimal[state]
        reward, state = simulator.get_next_reward_state(state, action)

        if state == simulator.terminal_state:
            steps_to_goal.append(step)
            break
    else:
        failed_starts += 1

print("Initial states tested:", len(simulator.initial_states))
print("Successful starts:", len(steps_to_goal))
print("Failed starts:", failed_starts)

if steps_to_goal:
    print("Average steps to goal:", sum(steps_to_goal) / len(steps_to_goal))
    print("Maximum steps to goal:", max(steps_to_goal))

Initial states tested: 71
Successful starts: 71
Failed starts: 0
Average steps to goal: 9.774647887323944
Maximum steps to goal: 13


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

I used ChatGPT to help write and understand the code for the random policy, Q-learning, epsilon-greedy policy, and policy evaluation. I used Python’s built-in random and csv modules for action selection and saving TSV files.